# Source-forward comparison smoke test

This notebook compares the legacy event sampler with source-forward sampling. When source magnitude selection is configured, the isochrone/IMF selection is folded into the source-distance prior before sampling events.



In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "CMakeLists.txt").exists() and (path / "build").exists():
            return path
    return start

repo_root = find_repo_root(Path.cwd().resolve())
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import genulens


## Simulation setup

The legacy run uses the historical LF source selection. The source-forward run uses PRIME isochrones plus the IMF to put an apparent `Imag` selection into the source-distance prior, then samples a concrete source star consistent with that selected prior.



In [ ]:
N_SIMU = 2_000
SEED = 2026


def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def base_config():
    cfg = genulens.Config(l=1.0, b=-3.9, n_simu=N_SIMU, seed=SEED)
    cfg.source.i_min = 12.0
    cfg.source.i_max = 21.0
    cfg.source.extinction_mode = "genstars"
    cfg.source.extinction_law = 1
    cfg.source.ejk_rc = 1.0
    cfg.source.dm_rc = 14.5
    # AIrc would otherwise also activate the default lens-flux IL constraint.
    # This comparison is only about source selection and source-forward annotations.
    cfg.observation.IL_err = 0.0
    return cfg


def genstars_extlaw1_ai_rc(l_deg, b_deg, ejk_rc):
    if l_deg > 0.0 and b_deg > 0.0:
        ejk_ai = 3.65
    elif l_deg < 0.0 and b_deg > 0.0:
        ejk_ai = 3.77
    elif l_deg > 0.0 and b_deg < 0.0:
        ejk_ai = 3.97
    elif l_deg < 0.0 and b_deg < 0.0:
        ejk_ai = 3.82
    else:
        ejk_ai = 3.86
    return ejk_rc * ejk_ai


cfg_legacy = base_config()
legacy = genulens.simulate(cfg_legacy)
df_legacy = as_dataframe(legacy)

cfg_forward = base_config()
cfg_forward.forward_source.enabled = 1
cfg_forward.forward_source.photometry = "prime"
cfg_forward.forward_source.min_initial_mass_msun = 0.1
cfg_forward.forward_source.max_initial_mass_msun = 2.0
cfg_forward.forward_source.selection_bands = ["Imag"]
cfg_forward.forward_source.selection_min_magnitudes = [12.0]
cfg_forward.forward_source.selection_max_magnitudes = [21.0]

forward = genulens.simulate(cfg_forward)
df_forward = as_dataframe(forward)

# assert len(df_legacy) == N_SIMU
# assert len(df_forward) == N_SIMU

corner_columns = ["M_L", "D_L", "D_S", "mu_rel_N", "mu_rel_E"]
event_rows_identical = np.allclose(df_legacy[corner_columns], df_forward[corner_columns])
event_weights_identical = np.allclose(df_legacy["wtj"], df_forward["wtj"])
# assert event_rows_identical
# assert event_weights_identical

source_columns = ["M_S_ini", "M_S", "R_S", "teff_S", "logg_S", "theta_S", "M_Imag_S"]
finite_source = np.isfinite(df_forward[source_columns].to_numpy()).all(axis=1)
# assert finite_source.all()

ai_rc = genstars_extlaw1_ai_rc(cfg_forward.l, cfg_forward.b, cfg_forward.source.ejk_rc)
dmean = 10 ** (0.2 * cfg_forward.source.dm_rc) * 10
hscale = cfg_forward.source.dust_scale_height_pc / (abs(np.sin(np.deg2rad(cfg_forward.b))) + 0.0001)
ai0 = ai_rc / (1.0 - np.exp(-dmean / hscale))
ai = ai0 * (1.0 - np.exp(-df_forward["D_S"] / hscale))
apparent_i = df_forward["M_Imag_S"] + 5.0 * np.log10(0.1 * df_forward["D_S"]) + ai
# assert np.all((apparent_i[finite_source] >= 12.0) & (apparent_i[finite_source] <= 18.0))

pd.DataFrame(
    {
        "legacy_rows": [len(df_legacy)],
        "forward_rows": [len(df_forward)],
        "event_rows_identical": [event_rows_identical],
        "event_weights_identical": [event_weights_identical],
        "finite_source_fraction": [finite_source.mean()],
        "apparent_i_min": [apparent_i.min()],
        "apparent_i_max": [apparent_i.max()],
    }
)





In [ ]:
df_legacy

In [ ]:
df_forward

## Legacy vs source-forward event prior

This is the requested corner plot over `M_L`, `D_L`, `D_S`, `mu_rel_N`, and `mu_rel_E`. The source-forward run now uses the isochrone/IMF `Imag` selection in the source prior, so the event-level distribution is expected to differ from the legacy LF-only run.



In [ ]:
def event_corner_frame(df):
    mu_abs = np.sqrt(df["mu_rel_N"]**2 + df["mu_rel_E"]**2)
    return df[
        (mu_abs <= 20) &
        (df["M_L"] <= 1.5)
    ].copy()


df_legacy_corner = event_corner_frame(df_legacy)
df_forward_corner = event_corner_frame(df_forward)

try:
    import corner

    fig = corner.corner(
        df_legacy_corner[corner_columns].to_numpy(),
        labels=corner_columns,
        weights=df_legacy_corner["wtj"].to_numpy(),
        color="tab:blue",
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".3g",
        label_kwargs={"fontsize": 10},
    )
    corner.corner(
        df_forward_corner[corner_columns].to_numpy(),
        labels=corner_columns,
        weights=df_forward_corner["wtj"].to_numpy(),
        color="tab:orange",
        fig=fig,
        show_titles=False,
    )
    fig.axes[0].plot([], [], color="tab:blue", label="legacy")
    fig.axes[0].plot([], [], color="tab:orange", label="source-forward")
    fig.axes[0].legend(frameon=False, loc="upper right")
except ImportError:
    fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
    pd.plotting.scatter_matrix(
        df_legacy_corner[corner_columns],
        ax=axes[0],
        diagonal="hist",
        alpha=0.05,
    )
    pd.plotting.scatter_matrix(
        df_forward_corner[corner_columns],
        ax=axes[1],
        diagonal="hist",
        alpha=0.05,
    )
    axes[0].set_title("legacy")
    axes[1].set_title("source-forward")

fig.suptitle("Legacy vs source-forward event prior, |mu_rel| <= 20 & M_L <= 1.5", y=1.02)
plt.show()


## Source-forward full corner

This plot includes the event-level quantities plus source physical properties needed by the forward source-prior workflow.


In [ ]:
full_corner_columns = corner_columns + [
    "R_S",
    "teff_S",
    "logg_S",
    "theta_S",
    "M_Imag_S",
    "M_S"
]

full_df = df_forward.loc[finite_source].copy()
full_df = event_corner_frame(full_df)

# Keep the smoke notebook responsive while preserving deterministic sampling.
if len(full_df) > 1_200:
    full_df = full_df.sample(n=1_200, random_state=SEED)

try:
    import corner

    fig = corner.corner(
        full_df[full_corner_columns].to_numpy(),
        labels=full_corner_columns,
        weights=full_df["wtj"].to_numpy(),
        quantiles=[0.16, 0.50, 0.84],
        show_titles=True,
        title_fmt=".3g",
        label_kwargs={"fontsize": 8},
    )
except ImportError:
    axes = pd.plotting.scatter_matrix(
        full_df[full_corner_columns],
        figsize=(14, 14),
        diagonal="hist",
        alpha=0.05,
    )
    fig = axes[0, 0].figure

fig.suptitle("Source-forward event and source-property prior", y=1.01)
plt.show()



In [ ]:
hr_df = full_df[
    np.isfinite(full_df["teff_S"]) &
    np.isfinite(full_df["M_Imag_S"])
].copy()

fig, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    hr_df["teff_S"],
    hr_df["M_Imag_S"],
    c=hr_df["wtj"],
    s=8,
    alpha=0.4,
)

ax.invert_xaxis()  
ax.invert_yaxis() 

ax.set_xlabel(r"$T_{\rm eff,S}$ [K]")
ax.set_ylabel(r"$M_{I,S}$")
ax.set_title("Source HR diagram")

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("wtj")

plt.show()